# Etiquetado heuristico — Tarea 1

Fragmenta el corpus de cada integrante, aplica etiquetado automatico por patrones
regex y produce dos salidas: el conjunto de entrenamiento y la muestra para
validacion humana.

Cambia `NOMBRE` y `ETIQUETAS_ASIGNADAS` en la celda de configuracion y ejecuta
todas las celdas en orden.

## Configuracion

In [ ]:
# Unica celda que debes editar
NOMBRE              = "Camilo"        # igual que en CARGA_TXT_MAIA_PROJECT
ETIQUETAS_ASIGNADAS = ["LIM", "CONC"]

# Referencia de asignaciones:
#   Jesus  -> ["INTRO", "BACK"]
#   Camilo -> ["LIM",   "CONC"]
#   Mateo  -> ["METH",  "RES"]
#   Sergio -> ["DISC",  "CONTR"]

INPUT_PATH        = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus.parquet"
OUTPUT_TRAIN_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_task1_train.parquet"
OUTPUT_VALID_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_task1_validacion.csv"
MAX_POR_ETIQUETA  = 2000
N_VALIDACION      = 200
SEED              = 42
MIN_PALABRAS      = 250
MAX_PALABRAS      = 1000

print(f"Miembro:             {NOMBRE}")
print(f"Etiquetas asignadas: {ETIQUETAS_ASIGNADAS}")
print(f"Entrada:             {INPUT_PATH}")
print(f"Entrenamiento:       {OUTPUT_TRAIN_PATH}")
print(f"Validacion:          {OUTPUT_VALID_PATH}")

## Instalacion

In [ ]:
!pip install -q pyarrow pandas tqdm

## Montaje de Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Patrones y funciones

Los patrones buscan indicadores de seccion en las primeras 5 lineas del
fragmento, donde suele aparecer el encabezado. Si no hay coincidencia,
el fragmento queda marcado como `UNKNOWN` y se descarta en el filtrado.

In [ ]:
import re

PATRONES = {
    "INTRO": [
        r"introducci[oó]n",
        r"planteamiento del problema",
        r"motivaci[oó]n",
        r"objetivo(s)? (del|de este)",
    ],
    "BACK": [
        r"trabajo(s)? relacionado(s)?",
        r"estado del arte",
        r"antecedentes",
        r"marco te[oó]rico",
    ],
    "METH": [
        r"metodolog[ií]a",
        r"m[eé]todo(s)?",
        r"dise[nñ]o experimental",
        r"materiales y m[eé]todos",
    ],
    "RES": [
        r"resultado(s)?",
        r"experimento(s)?",
        r"evaluaci[oó]n emp[ií]rica",
    ],
    "DISC": [
        r"discusi[oó]n",
        r"implicaciones",
        r"comparaci[oó]n con trabajos previos",
    ],
    "CONTR": [
        r"contribuci[oó]n(es)?",
        r"aporte(s)?",
        r"este trabajo propone",
        r"en este art[ií]culo se presenta",
    ],
    "LIM": [
        r"limitaci[oó]n(es)?",
        r"restricciones",
        r"trabajo futuro",
        r"l[ií]neas futuras",
    ],
    "CONC": [
        r"conclusi[oó]n(es)?",
        r"hallazgo(s)? principal(es)?",
        r"en resumen",
        r"a modo de conclusi[oó]n",
    ],
}

_COMPILADOS = {
    etiqueta: [re.compile(p, re.IGNORECASE) for p in patrones]
    for etiqueta, patrones in PATRONES.items()
}

N_LINEAS_ENCABEZADO = 5


def etiquetar_fragmento(texto):
    lineas = [l for l in texto.splitlines() if l.strip()]
    encabezado = " ".join(lineas[:N_LINEAS_ENCABEZADO])
    for etiqueta, patrones in _COMPILADOS.items():
        for patron in patrones:
            if patron.search(encabezado):
                return etiqueta
    return "UNKNOWN"


def fragmentar_texto(texto, min_palabras=250, max_palabras=1000):
    parrafos = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    fragmentos = []
    fragmento_actual = []
    palabras_acumuladas = 0

    for parrafo in parrafos:
        palabras_parrafo = len(parrafo.split())

        if palabras_parrafo > max_palabras:
            for oracion in re.split(r"(?<=[.!?])\s+", parrafo):
                palabras_oracion = len(oracion.split())
                if palabras_acumuladas + palabras_oracion > max_palabras:
                    if palabras_acumuladas >= min_palabras:
                        fragmentos.append("\n\n".join(fragmento_actual))
                    fragmento_actual = [oracion]
                    palabras_acumuladas = palabras_oracion
                else:
                    fragmento_actual.append(oracion)
                    palabras_acumuladas += palabras_oracion
            continue

        if palabras_acumuladas + palabras_parrafo > max_palabras:
            if palabras_acumuladas >= min_palabras:
                fragmentos.append("\n\n".join(fragmento_actual))
            fragmento_actual = [parrafo]
            palabras_acumuladas = palabras_parrafo
        else:
            fragmento_actual.append(parrafo)
            palabras_acumuladas += palabras_parrafo

    if fragmento_actual and palabras_acumuladas >= min_palabras:
        fragmentos.append("\n\n".join(fragmento_actual))

    return fragmentos


print(f"Etiquetas definidas: {list(PATRONES.keys())}")

## Cargar corpus

In [ ]:
import pandas as pd

df_corpus = pd.read_parquet(INPUT_PATH)

print(f"Documentos: {len(df_corpus):,}")
print(f"Columnas:   {df_corpus.columns.tolist()}")

df_corpus.head(2)

## Fragmentacion y etiquetado

In [ ]:
from tqdm.notebook import tqdm

registros = []

for _, fila in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc="Fragmentando"):
    for i, fragmento in enumerate(fragmentar_texto(fila["texto"], MIN_PALABRAS, MAX_PALABRAS)):
        registros.append({
            "doc_id":        fila["doc_id"],
            "fragmento_id":  f"{fila['doc_id']}_{i:04d}",
            "texto":         fragmento,
            "etiqueta_auto": etiquetar_fragmento(fragmento),
            "num_palabras":  len(fragmento.split()),
            "asignado_a":    NOMBRE,
        })

df_fragmentos = pd.DataFrame(registros)

print(f"Total fragmentos: {len(df_fragmentos):,}")
print()
print("Distribucion de etiquetas:")
print(df_fragmentos["etiqueta_auto"].value_counts().to_string())

## Filtrado por etiqueta

In [ ]:
# diagnostico: cuantos fragmentos hay por etiqueta antes de filtrar
dist = df_fragmentos["etiqueta_auto"].value_counts()
print("Distribucion completa:")
print(dist.to_string())
print()

for etiqueta in ETIQUETAS_ASIGNADAS:
    disponibles = dist.get(etiqueta, 0)
    tren_esperado = max(0, min(MAX_POR_ETIQUETA, disponibles - N_VALIDACION))
    val_esperada  = min(N_VALIDACION, disponibles)
    print(f"{etiqueta}: {disponibles:,} fragmentos → entrenamiento ~{tren_esperado:,} | validacion ~{val_esperada:,}")

total_tren = sum(max(0, min(MAX_POR_ETIQUETA, dist.get(e, 0) - N_VALIDACION)) for e in ETIQUETAS_ASIGNADAS)
print(f"\nTotal entrenamiento estimado: {total_tren:,} / {MAX_POR_ETIQUETA * len(ETIQUETAS_ASIGNADAS):,} objetivo")

In [ ]:
# conserva solo los fragmentos con etiquetas asignadas a este miembro
# los UNKNOWN quedan excluidos automaticamente
df_filtrado = df_fragmentos[
    df_fragmentos["etiqueta_auto"].isin(ETIQUETAS_ASIGNADAS)
].copy()

print(f"Fragmentos disponibles: {len(df_filtrado):,}")
print()
print("Distribucion:")
print(df_filtrado["etiqueta_auto"].value_counts().to_string())

## Separacion train / validacion

La validacion se extrae primero para garantizar que ningun fragmento
aparezca en ambos conjuntos. El entrenamiento se muestrea del pool restante.

In [ ]:
# paso 1: extraer muestra de validacion (200 por etiqueta)
df_validacion = (
    df_filtrado
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_VALIDACION, len(g)), random_state=SEED))
    .reset_index(drop=True)
)

# paso 2: excluir los ids de validacion del pool
ids_validacion = set(df_validacion["fragmento_id"])

# paso 3: samplear entrenamiento del pool restante (hasta 2000 por etiqueta)
df_entrenamiento = (
    df_filtrado[~df_filtrado["fragmento_id"].isin(ids_validacion)]
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.sample(n=min(MAX_POR_ETIQUETA, len(g)), random_state=SEED))
    .reset_index(drop=True)
)

print(f"Entrenamiento: {len(df_entrenamiento):,} fragmentos")
print(df_entrenamiento["etiqueta_auto"].value_counts().to_string())
print()
print(f"Validacion:    {len(df_validacion):,} fragmentos")
print(df_validacion["etiqueta_auto"].value_counts().to_string())

## Guardar resultados

In [ ]:
import os

# entrenamiento como parquet
df_entrenamiento.to_parquet(OUTPUT_TRAIN_PATH, index=False)
mb_train = os.path.getsize(OUTPUT_TRAIN_PATH) / (1024 ** 2)

# validacion como CSV para anotacion manual
planilla = pd.DataFrame({
    "doc_id":          df_validacion["doc_id"],
    "fragmento_id":    df_validacion["fragmento_id"],
    "texto_corto":     df_validacion["texto"].str[:300],
    "etiqueta_auto":   df_validacion["etiqueta_auto"],
    "etiqueta_humano": "",
    "anotador":        NOMBRE,
    "notas":           "",
})
planilla.to_csv(OUTPUT_VALID_PATH, index=False)

print(f"Entrenamiento: {OUTPUT_TRAIN_PATH}  ({mb_train:.1f} MB, {len(df_entrenamiento):,} filas)")
print(f"Validacion:    {OUTPUT_VALID_PATH}  ({len(planilla):,} filas)")
print()
print("Instrucciones de anotacion:")
print("  1. Abre el CSV en Google Sheets o Excel")
print("  2. Para cada fila, lee 'texto_corto' y verifica si 'etiqueta_auto' es correcta")
print(f"  3. Escribe la etiqueta correcta en 'etiqueta_humano' ({ETIQUETAS_ASIGNADAS + ['OTRO']})")
print("  4. Sube el archivo anotado de vuelta a Drive")

## Registro en DVC

In [ ]:
# El parquet de entrenamiento no va a GitHub directamente.
# Se copia a data/processed/task1/train/ y se registra con DVC.

nombre_lower = NOMBRE.lower()
destino = f"data/processed/task1/train/{nombre_lower}_task1_train.parquet"

print("Despues de ejecutar este notebook, corre en tu terminal local:")
print()
print(f"  cp {OUTPUT_TRAIN_PATH} {destino}")
print(f"  dvc add {destino}")
print(f"  git add {destino}.dvc .gitignore")
print(f"  git commit -m \"data: add {nombre_lower} task1 training set\"")
print( "  git push")